# Combined Pipeline: All Three Improvements

This notebook combines all three proposed improvements into a single end-to-end
pipeline and benchmarks the cumulative effect on separation quality.

| # | Improvement | Type | Notebook |
|---|---|---|---|
| 1 | Basic Pitch + Heuristic Pitch Splitting | Inference optimization | `05_basic_pitch_splitting.ipynb` |
| 2 | MIDI-Guided Post-Processing Audio Masking | Post-processing | `06_midi_guided_masking.ipynb` |
| 3 | NoteJitter Temporal Misalignment Augmentation | Training robustness | (in `src/training/augment.py`) |

### How they compose:

1. **Basic Pitch** transcribes the mix → single MIDI stream
2. **Pitch splitting** (median + hysteresis) divides notes into Guitar 1 / Guitar 2
3. Split notes **condition** the `best_conditioned.pt` model for separation
4. **MIDI masking** post-processes the output — muting hallucinated noise in silent regions
5. **Mixture consistency** correction ensures separated sources sum to the original mix

Improvement 3 (NoteJitter) operates at **training time**, making the model robust to
imperfect note alignments — exactly the kind produced by Basic Pitch at inference.
If you've fine-tuned with NoteJitter enabled, the conditioned model you load here
already benefits from it.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchaudio
import IPython.display as ipd
import museval

from src.models.factory import build_model
from src.models.apply import apply_model
from src.data.manifests import load_manifest
from src.inference.separate import create_tensor_for_segment
from src.utils.io import load_config

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1. Reusable components from Improvements 1 & 2

In [ ]:
from basic_pitch.inference import predict as bp_predict


def transcribe_mix(audio_path, onset_threshold=0.5, frame_threshold=0.3):
    """Transcribe a mix with Basic Pitch."""
    _, _, note_events = bp_predict(
        str(audio_path),
        onset_threshold=onset_threshold,
        frame_threshold=frame_threshold,
    )
    print(f"  Basic Pitch: {len(note_events)} notes")
    return note_events


def median_split_with_hysteresis(
    note_events, audio_length_samples, sr=44100,
    window_sec=0.5, hysteresis_semitones=2,
):
    """Split mixed notes into G1 (melody) / G2 (accompaniment) piano rolls."""
    T = audio_length_samples
    pitch_sum = np.zeros(T, dtype=np.float64)
    pitch_count = np.zeros(T, dtype=np.float64)

    for ev in note_events:
        s = int(ev[0] * sr)
        e = min(int(ev[1] * sr), T)
        pitch = int(ev[2])
        if s < e:
            pitch_sum[s:e] += pitch
            pitch_count[s:e] += 1

    win = max(1, int(window_sec * sr))
    kernel = np.ones(win) / win
    sm_p = np.convolve(pitch_sum, kernel, mode="same")
    sm_c = np.maximum(np.convolve(pitch_count, kernel, mode="same"), 1e-8)
    running_median = sm_p / sm_c

    global_median = float(np.median([int(ev[2]) for ev in note_events])) if note_events else 60.0
    running_median[sm_p == 0] = global_median

    g1 = np.zeros((128, T), dtype=np.float32)
    g2 = np.zeros((128, T), dtype=np.float32)
    last_asgn = {}

    for ev in sorted(note_events, key=lambda x: x[0]):
        s = int(ev[0] * sr)
        e = min(int(ev[1] * sr), T)
        pitch = int(ev[2])
        if s >= e or pitch < 0 or pitch >= 128:
            continue
        diff = pitch - running_median[min(s, T - 1)]
        if abs(diff) <= hysteresis_semitones:
            assignment = last_asgn.get(pitch, 1 if diff >= 0 else 2)
        else:
            assignment = 1 if diff > 0 else 2
        last_asgn[pitch] = assignment
        (g1 if assignment == 1 else g2)[pitch, s:e] = 1.0

    return torch.from_numpy(np.concatenate([g1, g2], axis=0))


def build_activity_mask(piano_roll_128, sr=44100, fade_ms=10):
    """Build smooth activity mask from a 128-note piano roll."""
    T = piano_roll_128.shape[-1]
    activity = (piano_roll_128.sum(dim=0) > 0).float()
    fade_samples = int(fade_ms * sr / 1000)
    if fade_samples < 1:
        return activity
    kernel = torch.ones(1, 1, fade_samples * 2 + 1) / (fade_samples * 2 + 1)
    smoothed = torch.nn.functional.conv1d(
        activity.unsqueeze(0).unsqueeze(0), kernel, padding=fade_samples
    ).squeeze()[:T]
    mask = torch.clamp(smoothed, 0.0, 1.0)
    return torch.maximum(mask, activity)


def apply_midi_masking(g1_audio, g2_audio, piano_roll, sr=44100, fade_ms=10):
    """Apply MIDI-guided masking to separated audio."""
    T = min(g1_audio.shape[-1], g2_audio.shape[-1], piano_roll.shape[-1])
    mask_g1 = build_activity_mask(piano_roll[:128, :T], sr=sr, fade_ms=fade_ms)
    mask_g2 = build_activity_mask(piano_roll[128:, :T], sr=sr, fade_ms=fade_ms)
    g1_m = g1_audio[:, :T] * mask_g1.unsqueeze(0)
    g2_m = g2_audio[:, :T] * mask_g2.unsqueeze(0)
    return g1_m, g2_m, mask_g1, mask_g2


def mixture_consistency(mix, g1, g2):
    """Enforce that g1 + g2 = mix by distributing the residual proportionally."""
    T = min(mix.shape[-1], g1.shape[-1], g2.shape[-1])
    mix_t = mix[:, :T]
    g1_t = g1[:, :T]
    g2_t = g2[:, :T]
    residual = mix_t - (g1_t + g2_t)
    total_energy = g1_t.abs() + g2_t.abs() + 1e-8
    g1_out = g1_t + residual * (g1_t.abs() / total_energy)
    g2_out = g2_t + residual * (g2_t.abs() / total_energy)
    return g1_out, g2_out


print("All pipeline components defined.")

## 2. Load models

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

def load_model(config_path, checkpoint_path, device):
    config = load_config(config_path)
    model = build_model(config["model"]["name"], config["model"].get("kwargs", {}))
    if checkpoint_path.exists():
        payload = torch.load(checkpoint_path, map_location=device, weights_only=False)
        state = payload.get("model_state_dict", payload)
        model.load_state_dict(state)
        print(f"Loaded: {checkpoint_path.name}")
    else:
        print(f"WARNING: {checkpoint_path} not found")
    model.to(device).eval()
    return model, config

model_cond,   cfg = load_model(REPO_ROOT / "configs" / "conditioned.yaml",   REPO_ROOT / "outputs" / "checkpoints" / "best_conditioned.pt",   device)
model_uncond, _   = load_model(REPO_ROOT / "configs" / "unconditioned.yaml", REPO_ROOT / "outputs" / "checkpoints" / "best_unconditioned.pt", device)

## 3. The full pipeline function

In [ ]:
def full_pipeline(model_cond, mix, sr, audio_path, device,
                  onset_threshold=0.5, frame_threshold=0.3,
                  window_sec=0.5, hysteresis_semitones=2, fade_ms=10):
    """End-to-end pipeline: Basic Pitch → Pitch Split → Conditioned Separation → MIDI Masking → Mixture Consistency."""

    # Step 1: Transcribe
    note_events = transcribe_mix(audio_path, onset_threshold, frame_threshold)

    # Step 2: Pitch split
    piano_roll = median_split_with_hysteresis(
        note_events, mix.shape[-1], sr=sr,
        window_sec=window_sec, hysteresis_semitones=hysteresis_semitones,
    )

    # Step 3: Conditioned separation
    ref = mix.mean(0)
    ref_mean, ref_std = ref.mean(), ref.std()
    normalized = (mix - ref_mean) / ref_std
    model_input = torch.cat((normalized, piano_roll), dim=0)

    with torch.no_grad():
        sources = apply_model(model_cond, model_input[None], progress=False, device=device)[0]
    sources = sources * ref_std + ref_mean
    g1_raw, g2_raw = sources[0].cpu(), sources[1].cpu()

    # Step 4: MIDI masking
    g1_masked, g2_masked, mask_g1, mask_g2 = apply_midi_masking(
        g1_raw, g2_raw, piano_roll, sr=sr, fade_ms=fade_ms,
    )

    # Step 5: Mixture consistency
    g1_final, g2_final = mixture_consistency(mix, g1_masked, g2_masked)

    return {
        "g1_raw": g1_raw, "g2_raw": g2_raw,
        "g1_masked": g1_masked, "g2_masked": g2_masked,
        "g1_final": g1_final, "g2_final": g2_final,
        "piano_roll": piano_roll,
        "mask_g1": mask_g1, "mask_g2": mask_g2,
    }


def separate_unconditioned(model, mix, device):
    """Baseline unconditioned separation."""
    ref = mix.mean(0)
    ref_mean, ref_std = ref.mean(), ref.std()
    normalized = (mix - ref_mean) / ref_std
    with torch.no_grad():
        sources = apply_model(model, normalized[None], progress=False, device=device)[0]
    sources = sources * ref_std + ref_mean
    return sources[0].cpu(), sources[1].cpu()


print("Pipeline function defined.")

## 4. Select test track and run

In [ ]:
manifest = load_manifest(REPO_ROOT / cfg["dataset"]["manifest"], resolve_root=REPO_ROOT)
test_entries = [e for e in manifest if e["split"] == cfg["dataset"]["test_split"]]
print(f"Test tracks: {len(test_entries)}")
for i, e in enumerate(test_entries):
    print(f"  [{i}] {e['track_name']}")

TRACK_IDX = 0
entry = test_entries[TRACK_IDX]
print(f"\nSelected: {entry['track_name']}")

mix, sr = torchaudio.load(entry["mix"])
g1_ref, _ = torchaudio.load(entry["sources"]["guitar1"])
g2_ref, _ = torchaudio.load(entry["sources"]["guitar2"])
print(f"Duration: {mix.shape[-1]/sr:.2f}s")

In [ ]:
print("=== Running full pipeline ===")
results = full_pipeline(model_cond, mix, sr, entry["mix"], device)

print("\n=== Running unconditioned baseline ===")
g1_uncond, g2_uncond = separate_unconditioned(model_uncond, mix, device)

print("\nDone.")

## 5. Metrics: progressive improvement breakdown

We compare four configurations to see the cumulative effect of each improvement:

1. **Unconditioned baseline** — `best_unconditioned.pt`, no notes, no masking
2. **+ Improvement 1** — Basic Pitch + conditioned model
3. **+ Improvement 2** — add MIDI masking
4. **+ Mixture consistency** — final correction

In [ ]:
def compute_bss_metrics(g1_ref, g2_ref, g1_est, g2_est, sr):
    min_len = min(g1_ref.shape[-1], g1_est.shape[-1])
    refs = np.stack([
        g1_ref[:, :min_len].T.numpy(),
        g2_ref[:, :min_len].T.numpy(),
    ], axis=0)
    ests = np.stack([
        g1_est[:, :min_len].T.numpy(),
        g2_est[:, :min_len].T.numpy(),
    ], axis=0)
    sdr, sir, isr, sar, _ = museval.metrics.bss_eval(
        refs, ests, compute_permutation=True,
        window=sr, hop=sr,
        framewise_filters=False, bsseval_sources_version=False,
    )
    return {
        "Guitar 1": {"SDR": float(np.nanmedian(sdr[0])), "SIR": float(np.nanmedian(sir[0])), "SAR": float(np.nanmedian(sar[0]))},
        "Guitar 2": {"SDR": float(np.nanmedian(sdr[1])), "SIR": float(np.nanmedian(sir[1])), "SAR": float(np.nanmedian(sar[1]))},
    }


configs = {
    "Unconditioned":     (g1_uncond,            g2_uncond),
    "+ BP Splitting":    (results["g1_raw"],     results["g2_raw"]),
    "+ MIDI Masking":    (results["g1_masked"],  results["g2_masked"]),
    "+ Mix Consistency": (results["g1_final"],   results["g2_final"]),
}

all_metrics = {}
for name, (g1e, g2e) in configs.items():
    all_metrics[name] = compute_bss_metrics(g1_ref, g2_ref, g1e, g2e, sr)

header = f"{'':20s}" + "".join(f"{n:>18s}" for n in configs)
print(header)
print("-" * len(header))
for source in ["Guitar 1", "Guitar 2"]:
    for metric in ["SDR", "SIR", "SAR"]:
        label = f"{source} {metric}"
        vals = "".join(f"{all_metrics[n][source][metric]:+14.2f} dB" for n in configs)
        print(f"{label:20s}{vals}")
    print()

## 6. Audio comparison

In [ ]:
min_len = min(mix.shape[-1], *(g.shape[-1] for g in
    [g1_uncond, g2_uncond, results["g1_final"], results["g2_final"]]))

print("=== Mix ===")
ipd.display(ipd.Audio(mix[:, :min_len].numpy(), rate=sr))

for guitar, ref, est_base, est_pipe in [
    ("Guitar 1", g1_ref, g1_uncond, results["g1_final"]),
    ("Guitar 2", g2_ref, g2_uncond, results["g2_final"]),
]:
    print(f"\n=== {guitar} — Reference ===")
    ipd.display(ipd.Audio(ref[:, :min_len].numpy(), rate=sr))
    print(f"=== {guitar} — Unconditioned Baseline ===")
    ipd.display(ipd.Audio(est_base[:, :min_len].numpy(), rate=sr))
    print(f"=== {guitar} — Full Pipeline (all improvements) ===")
    ipd.display(ipd.Audio(est_pipe[:, :min_len].numpy(), rate=sr))

## 7. Spectrogram comparison

In [ ]:
def plot_spectrogram(wav, sr, ax, title="", vmin=-60, vmax=0):
    mono = wav.mean(dim=0)[:min_len]
    spec = torch.stft(mono, n_fft=2048, hop_length=512,
                      window=torch.hann_window(2048), return_complex=True)
    mag_db = 20 * torch.log10(spec.abs().clamp(min=1e-8))
    ax.imshow(mag_db.numpy(), aspect="auto", origin="lower",
              extent=[0, min_len/sr, 0, sr/2], cmap="magma", vmin=vmin, vmax=vmax)
    ax.set_ylim(0, 8000)
    ax.set_title(title, fontsize=9)


fig, axes = plt.subplots(2, 3, figsize=(18, 8), sharex=True, sharey=True)

plot_spectrogram(g1_ref,              sr, axes[0, 0], "G1 — Reference")
plot_spectrogram(g1_uncond,           sr, axes[0, 1], "G1 — Unconditioned")
plot_spectrogram(results["g1_final"], sr, axes[0, 2], "G1 — Full Pipeline")
plot_spectrogram(g2_ref,              sr, axes[1, 0], "G2 — Reference")
plot_spectrogram(g2_uncond,           sr, axes[1, 1], "G2 — Unconditioned")
plot_spectrogram(results["g2_final"], sr, axes[1, 2], "G2 — Full Pipeline")

for ax in axes[-1]:
    ax.set_xlabel("Time (s)")
for ax in axes[:, 0]:
    ax.set_ylabel("Freq (Hz)")

plt.suptitle(f"Full Pipeline vs Baseline — {entry['track_name']}", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Bar chart: SDR improvement breakdown

In [ ]:
config_names = list(configs.keys())
x = np.arange(len(config_names))
width = 0.35

sdr_g1 = [all_metrics[n]["Guitar 1"]["SDR"] for n in config_names]
sdr_g2 = [all_metrics[n]["Guitar 2"]["SDR"] for n in config_names]

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, sdr_g1, width, label="Guitar 1", color="tab:blue", alpha=0.8)
bars2 = ax.bar(x + width/2, sdr_g2, width, label="Guitar 2", color="tab:orange", alpha=0.8)

ax.set_xlabel("Pipeline Stage")
ax.set_ylabel("SDR (dB)")
ax.set_title(f"SDR Improvement Breakdown — {entry['track_name']}")
ax.set_xticks(x)
ax.set_xticklabels(config_names, rotation=15)
ax.legend()
ax.grid(axis="y", alpha=0.3)

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f"{h:.1f}", xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords="offset points", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## 9. Batch evaluation over all test tracks

In [ ]:
batch_results = {name: [] for name in ["Unconditioned", "Full Pipeline"]}

for i, entry_i in enumerate(test_entries):
    print(f"[{i+1}/{len(test_entries)}] {entry_i['track_name']}", end=" ")
    mix_i, sr_i = torchaudio.load(entry_i["mix"])
    g1r, _ = torchaudio.load(entry_i["sources"]["guitar1"])
    g2r, _ = torchaudio.load(entry_i["sources"]["guitar2"])

    try:
        g1u, g2u = separate_unconditioned(model_uncond, mix_i, device)
        batch_results["Unconditioned"].append(
            compute_bss_metrics(g1r, g2r, g1u, g2u, sr_i))
    except Exception as exc:
        print(f"uncond FAILED: {exc}", end=" ")
        batch_results["Unconditioned"].append(None)

    try:
        res = full_pipeline(model_cond, mix_i, sr_i, entry_i["mix"], device)
        batch_results["Full Pipeline"].append(
            compute_bss_metrics(g1r, g2r, res["g1_final"], res["g2_final"], sr_i))
    except Exception as exc:
        print(f"pipeline FAILED: {exc}", end=" ")
        batch_results["Full Pipeline"].append(None)

    print("ok")

print("\n" + "=" * 65)
print("AGGREGATE RESULTS (median over all test tracks)")
print("=" * 65)
print(f"{'':20s} {'Unconditioned':>14s}  {'Full Pipeline':>14s}  {'Delta':>8s}")
print("-" * 60)
for source in ["Guitar 1", "Guitar 2"]:
    for metric in ["SDR", "SIR", "SAR"]:
        vals_u = [m[source][metric] for m in batch_results["Unconditioned"] if m]
        vals_p = [m[source][metric] for m in batch_results["Full Pipeline"] if m]
        med_u = np.nanmedian(vals_u) if vals_u else float("nan")
        med_p = np.nanmedian(vals_p) if vals_p else float("nan")
        delta = med_p - med_u
        label = f"{source} {metric}"
        print(f"{label:20s} {med_u:+10.2f} dB   {med_p:+10.2f} dB   {delta:+.2f}")
    print()

## 10. Improvement 3: NoteJitter Temporal Misalignment Augmentation

Unlike Improvements 1 and 2 (which operate at inference time), **NoteJitter** is a
**training-time augmentation** that makes the conditioned model robust to imperfect
note timing — exactly the kind of error introduced by Basic Pitch at inference.

### How it works

During training, for each active note in the piano roll:
- **Onset is shifted** by a small random amount (default: ±30ms)
- **Duration is scaled** by a random factor (default: ±15%)

This teaches the network that note boundaries are approximate, not pixel-perfect,
closing the gap between training (where ground-truth MIDI is exact) and inference
(where Basic Pitch estimates are approximate).

### Implementation

The `NoteJitter` augmentation is implemented in `src/training/augment.py` and is
applied during fine-tuning. If you loaded a checkpoint that was fine-tuned with
NoteJitter enabled, the conditioned model above already benefits from this
improvement.

To fine-tune with NoteJitter:

```bash
python scripts/train.py \
    --config configs/conditioned.yaml \
    --checkpoint outputs/checkpoints/best_conditioned.pt
```

NoteJitter is automatically applied when training with note conditioning enabled.